# Safety Lab: Vibe Coding Risks

This notebook shows how quickly generated code can look correct while still introducing problems.

It demonstrates two simple cases:
1. A runtime bug caused by missing input validation
2. A security bug caused by unsafe SQL string building

Each risky example is followed by a safer version.

## Example 1: Runtime Bug

A code generator might create a function that assumes every record has the expected keys.

That looks fine until real data is missing a field.

In [ ]:
def generated_total_cost(cart):
    # This version assumes every item has both price and quantity.
    return sum(item['price'] * item['quantity'] for item in cart)

sample_cart = [
    {'name': 'Notebook', 'price': 5, 'quantity': 2},
    {'name': 'Pen', 'price': 2}
]

try:
    print('Generated result:', generated_total_cost(sample_cart))
except Exception as error:
    print('Runtime bug:', type(error).__name__, '-', error)

In [ ]:
def safe_total_cost(cart):
    # Use defaults and validation so missing fields do not crash the program.
    total = 0
    for item in cart:
        price = item.get('price', 0)
        quantity = item.get('quantity', 1)
        total += price * quantity
    return total

print('Safe result:', safe_total_cost(sample_cart))

## Example 2: Unsafe Security Pattern

Another common mistake is building SQL queries by joining user input directly into the query string.

That can break the query and can open the door to SQL injection.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT)')
cursor.executemany('INSERT INTO users (username) VALUES (?)', [('alice',), ('bob',)])
connection.commit()

def unsafe_find_user(username):
    # This is risky because the SQL is built with string formatting.
    query = f"SELECT id, username FROM users WHERE username = '{username}'"
    print('Unsafe query:', query)
    return cursor.execute(query).fetchall()

print('Normal input:', unsafe_find_user('alice'))

try:
    # Even a quote character can break the query syntax.
    print('Problem input:', unsafe_find_user("o'reilly"))
except Exception as error:
    print('Security issue example:', type(error).__name__, '-', error)

In [ ]:
def safe_find_user(username):
    # Parameterized queries keep the input separate from the SQL command.
    query = 'SELECT id, username FROM users WHERE username = ?'
    print('Safe query:', query)
    return cursor.execute(query, (username,)).fetchall()

print('Safe normal input:', safe_find_user('alice'))
print('Safe quoted input:', safe_find_user("o'reilly"))

## Takeaways

Generated code still needs review.

Simple checks that help:
1. Validate inputs before using them
2. Handle missing data and edge cases
3. Never build SQL by concatenating raw user input
4. Prefer safe library features such as parameterized queries